# Libraries

Just like before, lets import the libraries we'll be using. In addition to our usual libraries, we'll also import the python file we used to save our data processing functions from our source code directory (src). We can import it like a module since we've added an __init__.py file within that folder alongside adding it to our system's path list.

In [1]:
import pandas as pd
import sys
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import mlflow
import json

sys.path.append(os.path.abspath(os.pardir))

from src.data_processing import *

# Dataset Processing

Here, we'll take a look at the processed dataset. Fortunately, we've already done the data processing in our data_processing python notebook, all we need to do now is to call the process_data() function we've prepared inside our data_processing python file.

In [26]:
dataset_path = os.path.join('..','datasets','Philippine Fake News Corpus.csv')
df = pd.read_csv(dataset_path)
train, test = process_data(
    df = df, 
    feature_col = 'Content',
    label_col = 'Label',
    random_state = 42,
    df_name = 'Philippine Fake News Corpus.csv'
)

display(train)
display(test)

,Content,Label
0,"[397, 398, 7972, 5154, 8, 2589, 3706, 5252, 10...",1
1,"[542, 7972, 1242, 2722, 4, 1841, 2654, 6506, 7...",0
2,"[542, 7972, 196, 7075, 29, 1582, 7953, 7926, 5...",0
3,"[397, 398, 7972, 220, 4354, 137, 7964, 7934, 1...",1
4,"[397, 398, 7972, 220, 4126, 73, 6316, 790, 795...",1
...,...,...
23678,"[311, 967, 7972, 220, 2905, 2615, 27, 8, 295, ...",0
23679,"[311, 846, 7972, 220, 3219, 2805, 1639, 7926, ...",0
23680,"[542, 7972, 220, 1133, 65, 3940, 137, 5609, 79...",0
23681,"[1716, 1715, 7972, 1055, 1063, 5485, 209, 65, ...",1


,Content,Label
0,"[4020, 4110, 295, 7972, 2129, 1433, 1346, 7941...",1
1,"[1716, 1715, 7972, 716, 150, 1523, 73, 64, 566...",1
2,"[397, 398, 7972, 7419, 358, 76, 426, 6153, 65,...",1
3,"[542, 7972, 220, 1475, 133, 8, 774, 27, 1148, ...",0
4,"[397, 398, 7972, 7233, 926, 7941, 220, 397, 39...",1
...,...,...
5916,"[4020, 4110, 295, 7972, 220, 2800, 201, 853, 3...",1
5917,"[542, 7972, 57, 2330, 205, 2467, 63, 1366, 450...",0
5918,"[397, 398, 7972, 7419, 347, 1290, 466, 5836, 7...",1
5919,"[1716, 1715, 7972, 57, 420, 190, 4242, 5202, 7...",1


Our function looks functional, separating our dataset into train and test in addition to encoding them into numeric representations that we can work with. However, before we can pass this to the model, it needs to be arranged into tensors.

In addition to this, we want to pass it by batches, as such, we'll have to configure a dataloader for our model.

# Dataloader

Our dataloader will convert our dataset into batches, a sample of the actual dataset that our model can learn with in increments. Before this, we'll have to convert it to tensors, the shapes of the tensors must be consistent, which means that we'll have to pad all the sequences to have the same length.

## Converting to Tensor and Padding

Before creating a custom dataset, we need to make our values into tensors with consistent shape. Making a consistent shape can be done through padding, luckily for us, pytorch already has a dedicated function for this. All we have to do now is to convert the values into tensors.

In [8]:
train.Content = train.Content.apply(lambda x: torch.tensor(x))
train.Label = train.Label.apply(lambda x: torch.tensor(x))

test.Content = test.Content.apply(lambda x: torch.tensor(x))
test.Label = test.Label.apply(lambda x: torch.tensor(x))

display(train.head())
display(test.head())

,Content,Label
0,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)
1,"[tensor(542), tensor(7972), tensor(1242), tens...",tensor(0)
2,"[tensor(542), tensor(7972), tensor(196), tenso...",tensor(0)
3,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)
4,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)


,Content,Label
0,"[tensor(4020), tensor(4110), tensor(295), tens...",tensor(1)
1,"[tensor(1716), tensor(1715), tensor(7972), ten...",tensor(1)
2,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)
3,"[tensor(542), tensor(7972), tensor(220), tenso...",tensor(0)
4,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)


After converting to tensors, we want to pad them. We can make use of the pad_sequence() function from pytorch to pad them to the max length in the dataset; however, if we do this separately for each dataset, we'll end up with two different lengths: one max for train and another for test. To resolve this, we'll have to briefly combine train and test, then pad them, afterwards, separate them when calling the custom Dataset class.

In [9]:
merged = pd.concat([
    train.Content,
    test.Content
])

padded = pad_sequence(merged, batch_first = True, padding_value = 3)
padded.shape

torch.Size([29604, 24186])

## Creating a Custom Dataset

Before we can make use of a dataloader, we need to create a custom dataset class that the dataloader can work with. Creating it is quite simple, as we simply need to create a pseudo-custom dataset class that has the basic dunder methods like indexing and len.

In [10]:
class FakeNewsDataset(Dataset):
    def __init__(self, feature, label):
        super().__init__()
        self.feature = feature
        self.label = label
        
    def __len__(self):
        return len(self.feature)
    
    def __getitem__(self, idx):
        feature = self.feature[idx]
        label = self.label[idx]
        
        return feature, label

In [11]:
fake_news_train = FakeNewsDataset(
    padded[:len(train),:],
    train.Label
)

fake_news_test = FakeNewsDataset(
    padded[len(train):, :],
    test.Label
)

In [12]:
fake_news_train[0], fake_news_train[0][0].shape

((tensor([ 397,  398, 7972,  ...,    3,    3,    3]), tensor(1)),
 torch.Size([24186]))

In [13]:
fake_news_test[0], fake_news_test[0][0].shape

((tensor([4020, 4110,  295,  ...,    3,    3,    3]), tensor(1)),
 torch.Size([24186]))

## Creating the Dataloader

In [14]:
train_loader = DataLoader(
    dataset = fake_news_train,
    batch_size = 32,
    shuffle = True
)

test_loader = DataLoader(
    dataset = fake_news_test,
    batch_size = 32,
    shuffle = True
)

# Model Architecture

Now that we've dealt with our dataset, all that's left is to design our model architecture. Before anything, we'll start with an embedding layer so that we can convert the indices into context vectors that the model can use to learn meaning. We'll use a similar architecture to a CNN, using 1 dimensional convolutional layers paired with max pooling layers. Afterwards, it will be passed to a flatten layer and finally to a linear layer for the final prediction.

Defining the parameters for the layers are quite simple, with the sole exception of the linear layer. The linear layer's input shape is wholly dependent on the output of the flattened output from the convolutional and pooling layers, but the way the output shape is defined in these layers are through a complex formula. One thing we can do to resolve this is to simulate a single forward pass through the layers until the flatten layer, then extract the size of that layer. We then plug that into the linear definition as the input shape. As this is only a simulation, we need to define this under a no gradient context so that the code recognizes to not build a gradient or treat it as a training input.

In [15]:
class FakeNewsDetector(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_id, conv_dim, kernel_size, max_seq):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, pad_id)
        self.conv1d = nn.Conv1d(embed_dim, conv_dim, kernel_size)
        self.pool1d = nn.MaxPool1d(kernel_size)
        self.flat = nn.Flatten()
        
        # Calculate Flattened shape
        with torch.no_grad():
            dummy = torch.zeros(1, max_seq, dtype = torch.int64)
            embed = self.embed(dummy)
            conv1d = self.conv1d(
                embed.transpose(1,2)
            )
            pool1d = self.pool1d(conv1d)
            flat = self.flat(pool1d)
            flattened_size = flat.size(1)
        
        self.fc = nn.Linear(flattened_size, 1)
        
    def forward(self, x):
        x = self.embed(x)
        x = x.transpose(1,2)
        x = self.conv1d(x)
        x = self.pool1d(x)
        x = self.flat(x)
        x = self.fc(x)
        return x

# Model Training

After defining the model, we'll train the model by defining the optimizer and the loss function. We will also log the model's parameters, dataset, and other metrics so that we can reproduce this same exact model in the future if ever we need to revisit this exact version of the model.

In [16]:
max_seq = fake_news_train[0][0].shape[0]
max_seq

24186

We'll store the model configurations under a dictionary so that we can just dump it in json format to a file using the json.dump() command later on for model logging and artifact tracking.

In [17]:
config = {
    'vocab_size': 8000,
    'embed_dim': 5,
    'pad_id': 3,
    'conv_dim': 4,
    'kernel_size': 5,
    'max_seq': max_seq,
}

model = FakeNewsDetector(**config)

Our model outputs raw logits so we'll use the Binary Cross Entropy with Logits as our loss function. Our optimizer will be Adam as it is a nice starting optimizer and is generally a great starting optimizer for most models.

In [18]:
optim = torch.optim.Adam(model.parameters(), lr = 0.001)
loss_func = nn.BCEWithLogitsLoss()

## Setting Up MLFlow Tracking

Before training, we have to setup our MLFlow tracking. We'll initialize our main database file under our models directory alongside our bce tokenizer on a separate folder labelled FakeNewsDetector. This FakeNewsDetector folder will store all the versions for our model.

In [14]:
model_path = os.path.join('..','models','FakeNewsDetector')

mlflow.set_tracking_uri('sqlite:///' + os.path.join(os.path.abspath(model_path),'mlflow.db'))

experiment = mlflow.get_experiment_by_name('FakeNewsDetector')

if experiment is None:
    experiment_id = mlflow.create_experiment(
        name = 'FakeNewsDetector',
        artifact_location = model_path
    )
else:
    experiment_id = experiment.experiment_id

curr_experiment = mlflow.set_experiment(experiment_id = experiment_id)
print(f'The current active experiment has been set to {curr_experiment.name} with id of {curr_experiment.experiment_id}.')

The current active experiment has been set to FakeNewsDetector with id of 1.


## Load Latest Model

In [15]:
finished_runs = mlflow.search_runs(
    filter_string = "status = 'FINISHED'",
    search_all_experiments = True,
    order_by = ['end_time DESC', 'metrics.Loss ASC']
)
finished_runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Loss,params.conv_dim,params.pad_id,params.kernel_size,params.embed_dim,params.max_seq,params.vocab_size,tags.mlflow.source.type,tags.version,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.source.name
0,f488585343114bbb87add6fd998d544a,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-30 09:41:40.357000+00:00,2026-05-30 09:48:27.933000+00:00,0.010168,4,3,5,5,24186,8000,NOTEBOOK,1.0,kayle,charming-cat-820,model_building.ipynb
1,859f1744b6ac44c8ad22f79ab61b658b,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-30 08:46:25.649000+00:00,2026-05-30 08:53:31.161000+00:00,0.075763,4,3,5,5,24186,8000,NOTEBOOK,None,kayle,sincere-cat-696,model_building.ipynb
2,225bddb08550475ba1bd78085bf05958,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:55:44.448000+00:00,2026-05-28 11:01:53.473000+00:00,2.548278,4,3,5,5,24186,8000,NOTEBOOK,None,kayle,caring-bee-173,model_building.ipynb


In [16]:
latest_model_id = finished_runs.run_id[0]
latest_model_loss = finished_runs.loc[finished_runs.run_id == latest_model_id, 'metrics.Loss'][0]
model.load_state_dict(torch.load(
    os.path.join(model_path, latest_model_id, 'artifacts', 'weights.pt')
))

print(f'Latest saved model is from Run ID "{latest_model_id}" with a loss of {round(latest_model_loss, 2)}')

Latest saved model is from Run ID "f488585343114bbb87add6fd998d544a" with a loss of 0.01


## Training Loop

In [ ]:
epochs = 3

with mlflow.start_run():
    
    # Log Parameters
    mlflow.log_params(config)
    
    for epoch in range(epochs):
        # Accumulate Loss
        epoch_loss = 0
        
        for batch in train_loader:
            X = batch[0]
            y = batch[1].to(torch.float32)
            
            logits = model(X)
            loss = loss_func(
                logits.reshape(-1),
                y.to(torch.float32)
            )
            
            epoch_loss += loss.item()
            
            optim.zero_grad()
            loss.backward()
            optim.step()
        
        # Track Loss
        mlflow.log_metric('Loss', epoch_loss, step = epoch)
        
        print(f'Epoch: {epoch} | Loss: {round(epoch_loss/len(train_loader), 4)}')
    
    # Temporary Weights File
    torch.save(
        model.state_dict(),
        'weights.pt'
    )
    
    # Temporary Config File
    with open('config.json', 'w') as f:
        json.dump(config, f, indent = 2)
    
    # Log temp files
    mlflow.log_artifact('weights.pt')
    mlflow.log_artifact('config.json')
    
    # Remove temporary files
    os.remove('weights.pt')
    os.remove('config.json')

Epoch: 0 | Loss: 0.0017
Epoch: 1 | Loss: 0.0003
Epoch: 2 | Loss: 0.0001


# Model Versioning

Having to store all our model runs on all versions under the same folder can make it seem really messy and unorganized; however, with mlflow, we can query the runs and search the model that we need. The only thing that is missing is the versions of the models we run. We can log this as a tag for our runs whenever we go to train the model. To do this, we will have to modify our training loop to include a tag. Additionally, we'll also remove the round function since our last model seems to have quite the low loss, which means rounding it may only give us zeros.

In [44]:
version = 1.0
epochs = 3

with mlflow.start_run():
    
    # Log Parameters
    mlflow.log_params(config)
    
    for epoch in range(epochs):
        # Accumulate Loss
        epoch_loss = 0
        
        for batch in train_loader:
            X = batch[0]
            y = batch[1].to(torch.float32)
            
            logits = model(X)
            loss = loss_func(
                logits.reshape(-1),
                y.to(torch.float32)
            )
            
            epoch_loss += loss.item()
            
            optim.zero_grad()
            loss.backward()
            optim.step()
        
        # Track Loss
        mlflow.log_metric('Loss', epoch_loss, step = epoch)
        
        print(f'Epoch: {epoch} | Loss: {epoch_loss/len(train_loader)}')
    
    # Temporary Weights File
    torch.save(
        model.state_dict(),
        'weights.pt'
    )
    
    # Temporary Config File
    with open('config.json', 'w') as f:
        json.dump(config, f, indent = 2)
    
    # Log temp files
    mlflow.log_artifact('weights.pt')
    mlflow.log_artifact('config.json')
    
    # Set model version tag
    mlflow.set_tag('version', version)
    
    # Remove temporary files
    os.remove('weights.pt')
    os.remove('config.json')

Epoch: 0 | Loss: 3.114748558275462e-05
Epoch: 1 | Loss: 1.960535336155368e-05
Epoch: 2 | Loss: 1.3722480454845063e-05


Now that we've finished a run, we can check if the version tag was properly added by searching all the runs.

In [19]:
mlflow.search_runs(
    filter_string = "status = 'FINISHED'",
    order_by = ['end_time DESC', 'metrics.Loss ASC'],
    search_all_experiments = True
)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Loss,params.conv_dim,params.pad_id,params.kernel_size,params.embed_dim,params.max_seq,params.vocab_size,tags.mlflow.source.type,tags.version,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.source.name
0,f488585343114bbb87add6fd998d544a,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-30 09:41:40.357000+00:00,2026-05-30 09:48:27.933000+00:00,0.010168,4,3,5,5,24186,8000,NOTEBOOK,1.0,kayle,charming-cat-820,model_building.ipynb
1,859f1744b6ac44c8ad22f79ab61b658b,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-30 08:46:25.649000+00:00,2026-05-30 08:53:31.161000+00:00,0.075763,4,3,5,5,24186,8000,NOTEBOOK,None,kayle,sincere-cat-696,model_building.ipynb
2,225bddb08550475ba1bd78085bf05958,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:55:44.448000+00:00,2026-05-28 11:01:53.473000+00:00,2.548278,4,3,5,5,24186,8000,NOTEBOOK,None,kayle,caring-bee-173,model_building.ipynb


With this, we can now create different model versions and properly store them accordingly. In addition, we can query all our runs and check to see which model performed the best. Right now, we only have 3 versions of our model, 2 of which are not properly given a version tag, and 1 with a version tag of 1.0. All of these used the same model architecture trained on the same dataset with the same model configurations and training parameters.

Now that we've completed majority of our code, we can work on making functions to call these code blocks, then implement them into our source code directory under a python file.

# Function Definitions

In [20]:
def feature_tensor_pad(train, test, feature_col: str, 
                       pad_idx: int = 3, batch_first: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
    """Generate a padded tensor using the feature column of both train and test.

    Args:
        train (_type_): The train dataset
        test (_type_): The test dataset
        feature_col (str): Name of the feature column
        pad_idx (int, optional): Value to use as pad in tensor. Defaults to 3.
        batch_first (bool, optional): Whether or not to have the batch size as the first dimension of the resulting tensor shape. Defaults to True.

    Returns:
        tuple[torch.Tensor, torch.Tensor]: Train and Test tensors with padded values.
    """
      
    merged = pd.concat([
        train[feature_col],
        test[feature_col]
    ])
    
    padded = pad_sequence(merged, batch_first = batch_first, padding_value = pad_idx)
    
    train, test = padded[:len(train), :], padded[len(train):, :]
    
    return train, test

In [21]:
def create_dataloaders(train, test, feature_col: str, label_col: str, **kwargs) -> tuple[DataLoader, DataLoader]:
    """Create the train and test dataloaders for the model.

    Args:
        train (_type_): The train dataset.
        test (_type_): The test dataset.
        feature_col (str): The name of the feature column for both datasets.
        label_col (str): The name of the label column for both datasets.

    Returns:
        tuple[DataLoader, DataLoader]: Dataloaders for both train and test set.
    """
    
    # Kwargs
    pad_idx = kwargs.get('pad_idx', 0)
    batch_first = kwargs.get('batch_first', True)
    batch_size = kwargs.get('batch_size', 32)
    shuffle = kwargs.get('shuffle', True)
    
    # Convert Feature and Labels to tensors
    train[feature_col] = train[feature_col].apply(lambda x: torch.tensor(x))
    train[label_col] = train[label_col].apply(lambda x: torch.tensor(x))
    
    test[feature_col] = test[feature_col].apply(lambda x: torch.tensor(x))
    test[label_col] = test[label_col].apply(lambda x: torch.tensor(x))
    
    train_x, test_x = feature_tensor_pad(
        train = train, 
        test = test,
        feature_col = feature_col,
        pad_idx = pad_idx,
        batch_first = batch_first
    )
    
    train_set = FakeNewsDataset(train_x, train[label_col])
    test_set = FakeNewsDataset(test_x, test[label_col])
    
    train_loader = DataLoader(
        dataset = train_set,
        batch_size = batch_size,
        shuffle = shuffle
    )
    
    test_loader = DataLoader(
        dataset = test_set,
        batch_size = batch_size,
        shuffle = shuffle
    )
    
    return train_loader, test_loader

In [22]:
def get_experiment(exp_name: str, uri_path: str, artifact_location: str = None) -> mlflow.entities.Experiment:
    """Retrieves an existing experiment with the given experiment name. If there is none, it creates an experiment name with the given artifact location.

    Args:
        exp_name (str): Name of the experiment to get. If there is none, it creates an experiment using this name.
        uri_path (str): Path to the sqlite database file that stores mlflow experiments and runs.
        artifact_location (str, optional): Location to store the artifact if an experiment is created. Defaults to None.

    Returns:
        mlflow.entities.Experiment: MLFlow experiment object used to track metrics and artifacts.
    """
    
    # Set Tracking Uri
    mlflow.set_tracking_uri('sqlite:///' + uri_path)
    
    # Get Existing Experiment
    experiment = mlflow.get_experiment_by_name(exp_name)
    
    # Create Experiment if None
    if experiment is None:
        experiment = mlflow.create_experiment(exp_name, artifact_location)
    
    return experiment

In [23]:
def load_latest_model(model: FakeNewsDetector, model_dir: str, 
                      return_runs: bool = False) -> pd.DataFrame:
    """Load the latest FakeNewsDetector model that was tracked by an mlflow experiment.

    Args:
        model (FakeNewsDetector): A FakeNewsDetector model that would be used to load the latest model.
        model_dir (str): The directory path to where the models are saved.
        return_runs (bool): Whether to return the records of finished runs. Defaults to False.
    """
    
    # Get all recent finished runs
    all_runs = mlflow.search_runs(
        filter_string = "status = 'FINISHED'",
        order_by = ['end_time DESC', 'metrics.Loss ASC'],
        search_all_experiments = True
    )
    
    # Get latest run id
    latest_run_id = all_runs.run_id[0]
    
    # Load latest model artifacts
    latest_artifact_dir = os.path.join(model_dir, latest_run_id, 'artifacts')
    
    model.load_state_dict(torch.load(
        os.path.join(latest_artifact_dir, 'weights.pt')
    ))
    
    return all_runs if return_runs else None

In [29]:
def train_model(model: FakeNewsDetector, model_config: dict, epochs: int, 
                datasets: tuple[pd.DataFrame, pd.DataFrame], model_version: float = None,
                batch_size: int = 32, lr: float = 0.001, optim = None, loss_func = None, 
                **kwargs) -> None:
    # Experiment Kwargs
    exp_name = kwargs.get('exp_name', 'FakeNewsDetector')
    model_dir = os.path.join('..','models',exp_name)
    
    uri_path = kwargs.get(
        'uri_path', 
        os.path.join(model_dir, 'mlflow.db')
    )
    
    artifact_location = kwargs.get('artifact_location', model_dir)
    
    # DataLoader Kwargs
    train, test = datasets[0], datasets[1]
    feature_col = kwargs.get('feature_col', 'Content')
    label_col = kwargs.get('label_col', 'Label')
    pad_idx = kwargs.get('pad_idx', 0)
    batch_first = kwargs.get('batch_first', True)
    shuffle = kwargs.get('shuffle', True)        
        
    # Set Experiment
    experiment = get_experiment(exp_name, uri_path, artifact_location)
    experiment = mlflow.set_experiment(experiment_id = experiment.experiment_id)
        
    # Process DataLoaders
    trainloader, testloader = create_dataloaders(
        train = train,
        test = test,
        feature_col = feature_col,
        label_col = label_col,
        pad_idx = pad_idx,
        batch_size = batch_size,
        batch_first = batch_first,
        shuffle = shuffle
    )
        
    # Load Latest Model
    all_runs = load_latest_model(model, model_dir, return_runs = True)
        
    # Model Optimizer, Loss, and Version
    if optim is None:
        optim = torch.optim.Adam(model.parameters(), lr)
        
    if loss_func is None:
        loss_func = nn.BCEWithLogitsLoss()
        
    if model_version is None:
        model_version = all_runs['tags.version'][0]
        
    # Main Loop
    with mlflow.start_run():
        
        # Log Model Configuration
        mlflow.log_params(model_config)
        
        # Training
        for epoch in epochs:
            epoch_loss = 0
            for batch in trainloader:
                X, y = batch[0], batch[1]
                
                logits = model(X)
                loss = loss_func(
                    logits.reshape(-1),
                    y.to(torch.float32)
                )
                
                epoch_loss += loss.item()
                
                optim.zero_grad()
                loss.backward()
                optim.step()
            
            # Log metric to current run
            mlflow.log_metric('train_loss', epoch_loss/len(trainloader), step = epoch)
        
        # Testing
        test_loss = 0
        for batch in testloader:
            X, y = batch[0], batch[1]
            
            logits = model(X)
            loss = loss_func(
                logits.reshape(-1),
                y.to(torch.float32)
            )
            
            test_loss += loss.item()
            
        mlflow.log_metric('test_loss', test_loss/len(testloader))
        
        # Create temporary files for artifact storing
        torch.save(
            model.state_dict(),
            'weights.pt'
        )
        
        with open('config.json', 'w') as f:
            json.dump(model_config, f, indent = 2)
            
        mlflow.log_artifact('weights.pt')
        mlflow.log_artifact('config.json')
        
        os.remove('weights.pt')
        os.remove('config.json')
        
        mlflow.set_tag('version', model_version)
    
    return